# NT120 — Pandas: đọc, kiểm tra và khám phá dữ liệu

Nội dung tách từ Lecture 03 — slides 47, 49, 54, 55, 57, 82.

Notebook tự tạo dữ liệu mô phỏng nếu `data/flows.csv` chưa có.

In [ ]:
# Slide 37 — Standard header
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 40)

# Works whether the notebook is opened from the project root or notebooks/.
PROJECT_ROOT = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents) if (folder / "requirements.txt").exists()),
    Path.cwd(),
)
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)
FLOW_PATH = DATA_DIR / "flows.csv"
print("pandas:", pd.__version__, "| numpy:", np.__version__)

In [ ]:
# Dữ liệu nhỏ mô phỏng cho các ví dụ Pandas bên dưới
raw = pd.DataFrame({
    "dst_port": [22, 80, 443, 8080, 53, 22, 80, 3389],
    "duration": [3.395, 61.5, 0.04, 120.0, 0.0, 4.1, 0.9, 0.02],
    "fwd_pkts": [8, 12, 1, 20, 2, 6, 3, 1],
    "bwd_pkts": [7, 10, 0, 19, 1, 4, 2, 0],
    "fwd_bytes": [6520, 22000, 0, 402110, 95, 7047, 250, 60],
    "bwd_bytes": ["6,520", "22,000", "0", "402,110", "95", "7,047", "250", "60"],
    "protocol": ["TCP", "UDP", "TCP", "TCP", "UDP", "TCP", "UDP", "TCP"],
    "label": ["BENIGN", "BENIGN", "PortScan", "BENIGN", "BENIGN", "BENIGN", "BENIGN", "PortScan"],
    "ttl": [64, "unknown", 128, 64, 255, 64, 64, 128],
    "start": pd.date_range("2026-09-14", periods=8, freq="D"),
})
raw.to_csv(FLOW_PATH, index=False)
raw.head()

## Đọc, kiểm tra, chọn và nhóm dữ liệu — slides 47, 49, 54, 55 và 57

In [ ]:
# Slide 47 — read_csv variations
df = pd.read_csv(FLOW_PATH, thousands=",")
head = pd.read_csv(FLOW_PATH, nrows=1000)
subset_at_load = pd.read_csv(
    FLOW_PATH,
    usecols=["dst_port", "duration", "label"],
    dtype={"dst_port": "int32"},
    na_values=["unknown", "?"],
)
df.dtypes

In [ ]:
# Slide 49 — six lines for a new CSV
print("shape:", df.shape)
df.info()
display(df.describe())
display(df["label"].value_counts(normalize=True))
display(df.isna().sum().sort_values(ascending=False).head(10))
df.head(3)

In [ ]:
# Slides 54–55 — selection and boolean masks
dur = df["duration"]
sub = df[["duration", "fwd_pkts", "label"]]
attacks = df[df["label"] != "BENIGN"]
long_udp = df[(df["protocol"] == "UDP") & (df["duration"] > 60)]
web_ports = df[df["dst_port"].isin([80, 443, 8080])]
m1 = df["duration"] > 1
m2 = df["protocol"] == "UDP"
display(df.loc[df["label"] == "PortScan", ["duration", "fwd_pkts"]])
display(df[m1 & m2])
print("both:", (m1 & m2).sum(), "| either:", (m1 | m2).sum(), "| not m1:", (~m1).sum())

In [ ]:
# Slide 57 — groupby and crosstab
cols = ["duration", "fwd_pkts", "fwd_bytes"]
display(df.groupby("label")[cols].agg(["mean", "median"]).round(2))
pd.crosstab(df["protocol"], df["label"])

## Appendix — lệnh tham khảo Pandas (slide 82)

Các cell dưới đây bổ sung những lệnh ngắn trong phần Appendix của slide. Chúng dùng dữ liệu mô phỏng đã tạo ở trên để có thể chạy thử.

In [ ]:
# Slide 82 — Pandas: inspect and select
# Cell khởi tạo: chạy cell này trước các cell tham khảo bên dưới.
ref = df.copy()

In [ ]:
# Kích thước: (số hàng, số cột) và tổng số hàng
print(ref.shape, len(ref))

In [ ]:
# Kiểu dữ liệu và số giá trị không thiếu
ref.info()

In [ ]:
# Kiểu dữ liệu của từng cột
display(ref.dtypes)

In [ ]:
# Thống kê mô tả các cột số
display(ref.describe())

In [ ]:
# Thống kê riêng cột duration
display(ref["duration"].describe())

In [ ]:
# Năm dòng đầu tiên
display(ref.head())

In [ ]:
# Ba dòng ngẫu nhiên, với seed cố định
display(ref.sample(3, random_state=42))

In [ ]:
# Tỉ lệ từng lớp
display(ref["label"].value_counts(normalize=True))

In [ ]:
# Số giá trị thiếu ở từng cột
display(ref.isna().sum())

In [ ]:
# Tỉ lệ giá trị thiếu ở từng cột
display(ref.isna().mean())

In [ ]:
# Dòng trùng, số giá trị riêng và các protocol
print(ref.duplicated().sum(), ref["protocol"].nunique(), ref["protocol"].unique())

In [ ]:
# Chọn một cột
display(ref["duration"])

In [ ]:
# Chọn nhiều cột
display(ref[["duration", "fwd_pkts"]])

In [ ]:
# Lọc với điều kiện boolean
display(ref[ref["duration"] > 0])

In [ ]:
# Lọc bằng query
display(ref.query("duration > 0"))

In [ ]:
# Lọc các port thuộc một danh sách
display(ref[ref["dst_port"].isin([22, 80])])

In [ ]:
# Chọn theo tên cột với loc
display(ref.loc[ref["duration"] > 0, ["duration", "protocol"]])

In [ ]:
# Chọn theo vị trí với iloc
ref.iloc[0:5, 0:3]